# W2D4 — Feature Engineering & Scaling — Lab

**Week 2 · Day 4 · Data Engineering & Preprocessing** · Lab

Yesterday you removed damage. Today you add signal.

A feature is a hypothesis about the problem, written as arithmetic. That is the whole difference
between engineering features and generating columns: if you cannot say the sentence the column
believes, it is not a feature, it is a coincidence.

The discipline today is **one at a time**. You will build five features and refit after each one, so
that at the end you can say which *idea* worked rather than that the batch did. Some of your ideas
will make the model worse. Those rows stay in the table — a feature that did not work is a result,
and deleting it is how a report becomes a sales pitch.

You'll leave with `engineered.parquet` and `lift_table.md`.

**Time budget:** ~115 minutes. Sections 1–2 are the lab; Section 3 is a stretch you may finish at home.

<div dir="rtl" align="right">

# الأسبوع ٢ اليوم ٤ — هندسة الخصائص والتقييس

**الأسبوع ٢ · اليوم ٤ · هندسة البيانات والمعالجة المسبقة** · معمل عملي

بالأمس أزلت الضرر، واليوم تضيف الإشارة.

والخاصية فرضية عن المسألة مكتوبة على شكل حساب. وهذا هو الفرق كله بين هندسة الخصائص وتوليد الأعمدة:
فإن لم تستطع قول الجملة التي يعتقدها العمود فهو ليس خاصية بل مصادفة.

والانضباط اليوم هو **واحدة واحدة**. فستبني خمس خصائص وتعيد التدريب بعد كل واحدة، حتى تستطيع في النهاية
أن تقول أي **فكرة** نجحت لا أن المجموعة نجحت. وبعض أفكارك سيجعل النموذج أسوأ، وتبقى تلك الصفوف في
الجدول — فالخاصية التي لم تنجح نتيجة، وحذفها هو ما يحوّل التقرير إلى عرض تسويقي.

ستخرج بملف `engineered.parquet` وملف `lift_table.md`.

**الزمن المتوقّع:** نحو ١١٥ دقيقة. القسمان الأول والثاني هما المعمل، والقسم الثالث إضافي يمكن إكماله في المنزل.

</div>

> **This is your lab notebook.** Work through the hints — they tell you what to do and where
> to look, not what to type. Stuck for more than ten minutes on one task? Open the `_guided`
> version. That is not cheating; sitting stuck in silence is the only mistake. The full
> solution is released at the end of the day.

<div dir="rtl" align="right">

> **هذا دفتر المعمل الخاص بك.** اعمل وفق الإرشادات — فهي تخبرك بما يجب فعله وأين تبحث، لا بما
> تكتبه حرفيًا. إذا توقّفت أكثر من عشر دقائق عند مهمة واحدة فافتح نسخة `_guided`؛ هذا ليس غشًّا،
> والخطأ الوحيد هو أن تبقى متوقّفًا بصمت. ويُنشر الحل الكامل في نهاية اليوم.

</div>

## Learning objectives

By the end of this lab you can:

- Say the sentence a feature believes, before you write the arithmetic.
- Build features with the four moves that produce most of them: products and ratios, differences,
  per-group aggregates, and time deltas.
- Measure one feature's contribution on its own, and keep the negative results.
- Explain what Min-Max, Standard and Robust each assume, and which one survives an outlier.
- Say why scaling changes a linear model's geometry and changes nothing at all for a tree.
- Fit a scaler on the training data only, and check that you did.

<div dir="rtl" align="right">

## أهداف التعلّم

بنهاية هذا المعمل ستكون قادرًا على:

- قول الجملة التي تعتقدها الخاصية قبل كتابة الحساب.
- بناء الخصائص بالحركات الأربع التي تُنتج معظمها: الضرب والنسب، والفروق، والتجميعات لكل مجموعة، وفروق الزمن.
- قياس مساهمة الخاصية الواحدة وحدها، والاحتفاظ بالنتائج السالبة.
- شرح ما يفترضه كل من Min-Max وStandard وRobust، وأيّها يصمد أمام قيمة شاذّة.
- بيان لماذا يغيّر التقييس هندسة النموذج الخطّي ولا يغيّر شيئًا للشجرة.
- تدريب المُقيِّس على بيانات التدريب وحدها، والتحقّق أنك فعلت ذلك.

</div>

## About the data

**Dataset:** `messy_sales`, via **yesterday's `cleaned.parquet`** — 4,880 rows × 11 columns

This is not the raw file. It is what you produced in W2D3: the 120 duplicate rows gone, `city`
collapsed from fourteen spellings to five, both date columns parsed, `discount_pct` imputed with its
median alongside a `discount_was_missing` indicator, and — the important one — **`commission_paid`
dropped**, because it was 3% of the target paid after the fact.

If you did not finish yesterday, `load_artefact` falls back to the reference copy in
`shared/solutions_cache/` and prints a note saying so. You are not blocked.

The target is `revenue`. Your starting score is the honest 0.79 you ended yesterday on, not the
1.0000 the leak was giving you.

**Watch out:** one of the five features you build today will recover most of what dropping the
commission cost you. That is not a contradiction and it is not the leak coming back — it is the
difference between information that arrives from the future and information that was in front of you
all along. Section 3 makes you prove which one you have.

<div dir="rtl" align="right">

## عن البيانات

**مجموعة البيانات:** `messy_sales` عبر ملف **الأمس `cleaned.parquet`** — ٤٬٨٨٠ صفًا × ١١ عمودًا

هذا ليس الملف الخام، بل ما أنتجته في اليوم الثالث: الصفوف المكرّرة المئة والعشرون محذوفة، وعمود `city`
مجموع من أربع عشرة تهجئة إلى خمس، وعمودا التاريخ محلَّلان، والعمود `discount_pct` معوَّض بوسيطه مع
عمود مؤشّر `discount_was_missing`، والأهم: **العمود `commission_paid` محذوف** لأنه كان ٣٪ من الهدف
تُدفع بعد الواقعة.

وإن لم تُكمل الأمس فإن `load_artefact` ترجع إلى النسخة المرجعية في `shared/solutions_cache/` وتطبع
ملاحظة بذلك، فأنت غير متعطّل.

والهدف هو `revenue`. ونتيجتك الابتدائية هي ٠٫٧٩ الصادقة التي انتهيت إليها بالأمس، لا ١٫٠٠٠٠ التي كان
التسريب يعطيك إياها.

**انتبه:** إحدى الخصائص الخمس التي تبنيها اليوم ستستعيد معظم ما كلّفك حذف العمولة. وهذا ليس تناقضًا
ولا عودةً للتسريب، بل هو الفرق بين معلومة تأتي من المستقبل ومعلومة كانت أمامك من البداية. والقسم
الثالث يجعلك تُثبت أيّهما لديك.

</div>

## Setup

Run the cell below first. It loads yesterday's artefact and defines `score_frame` again — the same
plain `LinearRegression` on the same 80/20 split with the same seed, so that every number you compare
today differs only by the feature you added.

`n_estimators` for the forest in Section 2 is capped at 60. The lab is about features, not about
training time.

<div dir="rtl" align="right">

## الإعداد

شغّل الخلية التالية أولًا. تُحمّل مخرجات الأمس وتُعرّف `score_frame` من جديد — الانحدار الخطّي البسيط
نفسه على التقسيم نفسه بالبذرة نفسها، فلا يختلف أي رقمين تقارنهما اليوم إلا بالخاصية التي أضفتها.

وعدد الأشجار في القسم الثاني محدود بستّين، فالمعمل عن الخصائص لا عن زمن التدريب.

</div>

In [ ]:
# === AIEP portable setup — works locally (Miniconda + uv) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    # A clone that never ran `uv pip install -e shared/` still has the package on disk —
    # use it before reaching for the network. Colab (no clone) falls through to pip.
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, versions
from aiep.data import load_artefact
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, report

ensure("scikit-learn", "matplotlib", "pyarrow")
seed_everything(42)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

from aiep.viz import use_course_style, PALETTE
use_course_style()

CATEGORICAL = ["city", "channel", "customer_tier"]
TARGET = "revenue"
N_TREES = 60          # capped on purpose — this lab is about features, not training time


def build_matrix(frame, numeric_cols):
    """One-hot the categoricals, keep the numerics, return a float matrix."""
    return pd.get_dummies(
        frame[list(numeric_cols) + CATEGORICAL], columns=CATEGORICAL, drop_first=True
    ).astype("float64")


def split(frame, numeric_cols):
    """The one split this whole lab uses. Same seed everywhere, so scores are comparable."""
    return train_test_split(
        build_matrix(frame, numeric_cols), frame[TARGET], test_size=0.2, random_state=42
    )


def score_frame(frame, numeric_cols, model=None):
    """Fit and return R2 on the held-out 20%. Defaults to the same LinearRegression."""
    X_train, X_test, y_train, y_test = split(frame, numeric_cols)
    model = LinearRegression() if model is None else model
    return model.fit(X_train, y_train).score(X_test, y_test)


cleaned = pd.read_parquet(load_artefact("cleaned.parquet"))

print(f"{cleaned.shape[0]:,} rows x {cleaned.shape[1]} columns")
print(f"columns: {list(cleaned.columns)}")
print("\n", versions())

## Section 1 — Warm-up: the five rows from this morning  (≈25 min)

Everything in this section already works, and every number it prints is a number from the slides.
Run it, and check them off against your notes — if one of them differs, one of us is wrong and it is
worth finding out which.

The table is the one from the session:

| row | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| `total_charges` | 1440 | 2040 | 480 | 5760 | **0** |
| `tenure_months` | 12 | 24 | 6 | 48 | **0** |

Two columns three orders of magnitude apart, and a fifth row that will break a division later.

<div dir="rtl" align="right">

## القسم الأول — التهيئة: الصفوف الخمسة من هذا الصباح (نحو ٢٥ دقيقة)

كل ما في هذا القسم يعمل أصلًا، وكل رقم يطبعه رقم من الشرائح. شغّله وطابقه على ملاحظاتك — فإن اختلف
رقم فأحدنا مخطئ، ويستحق الأمر معرفة أيّنا.

والجدول هو جدول الجلسة نفسه: عمودان يفرقهما ثلاث مراتب عشرية، وصف خامس سيُعطّل قسمةً لاحقًا.

</div>

In [ ]:
X = np.array([[1440., 12], [2040, 24], [480, 6], [5760, 48], [0, 0]])

raw_distance = np.linalg.norm(X[0] - X[1])
print(f"charges difference: {X[1, 0] - X[0, 0]:.0f}   tenure difference: {X[1, 1] - X[0, 1]:.0f}")
print(f"distance between rows 1 and 2: {raw_distance:.2f}")
print(f"the whole contribution of the tenure column: {raw_distance - 600:.2f}")
print("\nTwo parts in ten thousand. Tenure is present in the arithmetic and absent from the result.")

Now Min-Max both columns and recompute the same distance. The rule is `(x − min) / (max − min)`.

Watch row 1: it lands on **(0.25, 0.25)**. The two columns were saying the same thing about that
customer all along, and the raw numbers hid it.

<div dir="rtl" align="right">

والآن قيّس العمودين بـ Min-Max وأعد حساب المسافة نفسها. والقاعدة `(x − min) / (max − min)`.

وراقب الصف الأول: فهو يقع على **(٠٫٢٥، ٠٫٢٥)**. فالعمودان كانا يقولان الشيء نفسه عن ذلك العميل من
البداية، والأرقام الخام أخفت ذلك.

</div>

In [ ]:
minmax = MinMaxScaler().fit_transform(X)

print("total_charges scaled:", np.round(minmax[:, 0], 4).tolist())
print("tenure_months scaled:", np.round(minmax[:, 1], 4).tolist())
print(f"\nrow 1 is now ({minmax[0, 0]:.2f}, {minmax[0, 1]:.2f})")

scaled_distance = np.linalg.norm(minmax[0] - minmax[1])
print(f"\nthe differences after scaling: {minmax[1, 0] - minmax[0, 0]:.4f} and "
      f"{minmax[1, 1] - minmax[0, 1]:.4f}")
print(f"distance between rows 1 and 2: {scaled_distance:.4f}")
print("\nNow tenure contributes MORE than charges does.")
print("The data did not change. The geometry did.")

`StandardScaler` centres on the mean and divides by the standard deviation, so the result is a
**z-score**: how many deviations from the mean this row sits. Mean zero, deviation one, and — unlike
Min-Max — no upper bound.

<div dir="rtl" align="right">

يُركِّز `StandardScaler` على المتوسط ويقسم على الانحراف المعياري، فيكون الناتج **درجة معيارية**: أي كم
انحرافًا يبعد هذا الصف عن المتوسط. متوسطه صفر وانحرافه واحد، وبخلاف Min-Max لا حدّ أعلى له.

</div>

In [ ]:
print(f"mean of total_charges: {X[:, 0].mean():.1f}")
print(f"deviations from it:    {(X[:, 0] - X[:, 0].mean()).astype(int).tolist()}")
print(f"standard deviation:    {X[:, 0].std():.2f}")

z = StandardScaler().fit_transform(X)
print(f"\nthe five z-scores:     {np.round(z[:, 0], 4).tolist()}")
print("\nRow 4 sits at 1.87, and nothing stops it being 12 if the value were larger.")

### Task 1.1 — the mis-keyed value

A sixth row arrives: `total_charges` of **99,999**, a data-entry error on an enterprise account.

Predict before you run it: which of the three scalers survives it? Then measure. The measurement is
**how much of the span of the five real values is left** after refitting each scaler on six rows.

**Change one thing:** replace 99,999 with 9,999 and re-run. The ranking stays; the sizes change.
That is worth seeing, because "Robust is best" is not the lesson — "Robust is more resistant, and by
how much depends on the outlier" is.

<div dir="rtl" align="right">

### المهمة ١٫١ — القيمة المُدخلة خطأً

يصل صف سادس: قيمة `total_charges` فيه **٩٩٬٩٩٩**، وهو خطأ إدخال في حساب مؤسسي.

توقّع قبل التشغيل: أيّ المُقيِّسات الثلاثة يصمد أمامه؟ ثم قِس. والقياس هو **كم يبقى من مدى القيم
الخمس الحقيقية** بعد إعادة تدريب كل مُقيِّس على ستة صفوف.

**غيّر شيئًا واحدًا:** استبدل ٩٩٬٩٩٩ بـ ٩٬٩٩٩ وأعد التشغيل. فالترتيب يبقى والمقادير تتغيّر. ويستحق
هذا المشاهدة، لأن الدرس ليس «Robust هو الأفضل» بل «Robust أكثر مقاومة، وبمقدار يتوقّف على القيمة الشاذّة».

</div>

In [ ]:
OUTLIER = 99_999.0            # <- change to 9_999 and re-run
X6 = np.vstack([X, [OUTLIER, 60]])

rows = []
for name, scaler in [("Min-Max", MinMaxScaler()),
                     ("Standard", StandardScaler()),
                     ("Robust", RobustScaler())]:
    alone = scaler.fit_transform(X)[:, 0]
    with_outlier = scaler.fit_transform(X6)[:5, 0]     # refit on six, look at the five
    rows.append({"scaler": name,
                 "span alone": np.ptp(alone),
                 "span with the outlier": np.ptp(with_outlier),
                 "collapse": np.ptp(alone) / np.ptp(with_outlier)})

collapse = pd.DataFrame(rows)
print(collapse.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

print("\nMin-Max squeezed every real customer into the bottom 5.8% of the range.")
print("Standard was no better — slightly worse: the mean and the deviation both moved.")
print("Robust kept 24x more of the structure than Min-Max did.")
print("\nAnd the honest part: Robust's median and IQR moved too. It collapsed 2.6x, not 1x.")
print("More resistant, not immune. That is what the word means.")

### Task 1.2 — the ratio, and the row that breaks it

Now the engineering half of the session. The hypothesis: *what a customer pays per month of their
tenure says something the raw total does not.* Written as arithmetic, that is
`total_charges / tenure_months`.

Four of the five rows give you a clean number. Row 5 is `0 / 0`, which is not infinity — it is
undefined.

<div dir="rtl" align="right">

### المهمة ١٫٢ — النسبة والصف الذي يُعطّلها

والآن النصف الهندسي من الجلسة. والفرضية: *أن ما يدفعه العميل شهريًا على مدّة اشتراكه يقول شيئًا لا
يقوله الإجمالي الخام*. ومكتوبةً حسابًا تكون `total_charges / tenure_months`.

وأربعة من الصفوف الخمسة تعطيك رقمًا نظيفًا، أما الصف الخامس فهو `0 / 0`، وهذا ليس لا نهاية بل غير معرَّف.

</div>

In [ ]:
charges, tenure = X[:, 0], X[:, 1]

# Divide only where the denominator is non-zero; the rest stays NaN rather than inf.
avg_monthly_spend = np.divide(charges, tenure, out=np.full_like(charges, np.nan),
                              where=tenure > 0)

print("avg_monthly_spend:", [f"{v:.0f}" if np.isfinite(v) else "NaN"
                             for v in avg_monthly_spend])
print("\nRow 5 is NaN, not 0 and not inf.")
print("It is yesterday's customer: no billing history, because they have not been billed.")
print("So it gets yesterday's remedy — leave it missing and add the indicator.")
print("fillna(0) would claim they spend nothing, and that is not what we know about them.")

## Section 2 — Core: five features, measured one at a time  (≈60 min)

1. Establish the baseline. One number, written down.
2. Build five features, **refitting after each one**, and record every score.
3. Write `lift_table.md` — feature, hypothesis, before, after, delta. Keep the negatives.
4. Scale three ways and refit. Notice what does *not* happen.
5. Fit a forest on scaled and unscaled data, and explain why the scores are the same.

The rule for task 2 is the whole point of the lab: **one feature, one refit, one number.** Adding
five features and comparing one before-and-after tells you the batch helped. It does not tell you
which idea was any good, and it cannot tell you which one to drop.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: خمس خصائص تُقاس واحدة واحدة (نحو ٦٠ دقيقة)

١. ثبّت خط الأساس. رقم واحد يُكتب.
٢. ابنِ خمس خصائص **مع إعادة التدريب بعد كل واحدة**، وسجّل كل نتيجة.
٣. اكتب ملف `lift_table.md`: الخاصية والفرضية والنتيجة قبل وبعد والفرق. واحتفظ بالسالب.
٤. قيّس بثلاث طرائق وأعد التدريب. ولاحظ ما **لا** يحدث.
٥. درّب غابة عشوائية على بيانات مُقيَّسة وغير مُقيَّسة، واشرح لماذا تتساوى النتيجتان.

وقاعدة المهمة الثانية هي جوهر المعمل كله: **خاصية واحدة، وإعادة تدريب واحدة، ورقم واحد.** فإضافة خمس
خصائص ومقارنة قبل وبعد واحدة تخبرك أن المجموعة ساعدت، ولا تخبرك أي فكرة كانت جيّدة، ولا تستطيع أن
تخبرك أيّها تحذف.

</div>

### Task 2.1 — the baseline

Fit the model on the cleaned data exactly as it arrived. Every number in the rest of this lab is
measured against this one, so write it down somewhere you will still have it in twenty minutes.

<div dir="rtl" align="right">

### المهمة ٢٫١ — خط الأساس

درّب النموذج على البيانات المُنظَّفة كما وصلت بالضبط. فكل رقم في بقيّة هذا المعمل يُقاس بالنسبة إلى
هذا الرقم، فاكتبه في مكان يبقى معك بعد عشرين دقيقة.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) The numeric columns you start with are quantity, unit_price, discount_pct
#    and the discount_was_missing indicator you built yesterday.
# 2) Score the frame as-is with score_frame and print it to four decimals.
# Search: "sklearn LinearRegression score r2"
# https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html
#
# ١) الأعمدة الرقمية التي تبدأ بها هي quantity وunit_price وdiscount_pct
#    ومؤشّر discount_was_missing الذي بنيته بالأمس.
# ٢) قيّم الجدول كما هو بـ score_frame واطبع النتيجة بأربع منازل عشرية.
# ابحث عن: "sklearn LinearRegression score r2"
# ────────────────────────────────────────────────────────────────────

BASE_NUMERIC = ["quantity", "unit_price", "discount_pct", "discount_was_missing"]
print(f"baseline R2 with no engineered features: {baseline_r2:.4f}")

### Task 2.2 — five features, five refits

Five features. For each one, **say the sentence first**, then write the arithmetic, then refit and
record the score. The sentence is not decoration: it is what makes the result interpretable when the
delta comes back negative.

| # | Feature | The sentence it believes | The move |
|---|---|---|---|
| 1 | `gross_order_value` | what the order was worth before any discount drives what it earns | product |
| 2 | `discount_amount` | the money given away matters in riyals, not as a percentage | difference |
| 3 | `unit_price_vs_city` | a price is expensive or cheap relative to its own city, not absolutely | per-group aggregate |
| 4 | `shipping_days` | orders that take longer to ship are a different kind of order | time delta |
| 5 | *(yours)* | — | your choice |

Feature 5 is yours to invent. Say its sentence in the markdown cell below the code before you look at
its delta — committing to the hypothesis first is what makes the measurement mean anything.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — خمس خصائص وخمس إعادات تدريب

خمس خصائص. ولكل واحدة **قل الجملة أولًا** ثم اكتب الحساب ثم أعد التدريب وسجّل النتيجة. والجملة ليست
زخرفة، بل هي ما يجعل النتيجة قابلة للتفسير حين يعود الفرق سالبًا.

| # | الخاصية | الجملة التي تعتقدها | الحركة |
|---|---|---|---|
| ١ | `gross_order_value` | أن قيمة الطلب قبل أي خصم هي ما يحدّد ما يكسبه | ضرب |
| ٢ | `discount_amount` | أن المال المتنازَل عنه يهمّ بالريالات لا بالنسبة المئوية | فرق |
| ٣ | `unit_price_vs_city` | أن السعر غالٍ أو رخيص بالنسبة إلى مدينته لا بالمطلق | تجميع لكل مجموعة |
| ٤ | `shipping_days` | أن الطلبات التي يطول شحنها نوع مختلف من الطلبات | فرق زمني |
| ٥ | *(خاصيتك)* | — | اختيارك |

والخاصية الخامسة من ابتكارك. قل جملتها في خلية الشرح تحت الشيفرة قبل أن تنظر إلى فرقها — فالالتزام
بالفرضية أولًا هو ما يجعل للقياس معنى.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Work on a copy called `engineered`. Add ONE column, then score, then add the
#    next — the running list of numeric columns grows by one each time.
# 2) gross_order_value = unit_price x quantity. discount_amount = that x discount_pct.
# 3) unit_price_vs_city needs the mean unit_price of each city aligned back onto every
#    row. Look for the groupby method that returns a column, not a summary.
# 4) shipping_days is a difference between two datetime columns — you want it as a
#    number of days, not as a timedelta.
# 5) Collect (name, sentence, before, after) into a list as you go. You need all four
#    for the lift table, and reconstructing them afterwards is how they get wrong.
# Search: "pandas groupby transform mean"
# https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.transform.html
#
# ١) اعمل على نسخة اسمها `engineered`. أضف عمودًا **واحدًا** ثم قيّم ثم أضف التالي —
#    فقائمة الأعمدة الرقمية تكبر بواحد كل مرة.
# ٢) العمود gross_order_value هو unit_price في quantity، وdiscount_amount هو حاصلهما
#    في discount_pct.
# ٣) ويحتاج unit_price_vs_city متوسط سعر الوحدة في كل مدينة محاذًى على كل صف.
#    ابحث عن دالة التجميع التي تُعيد عمودًا لا ملخّصًا.
# ٤) والعمود shipping_days فرق بين عمودي تاريخ، وتريده عددًا من الأيام لا فترة زمنية.
# ٥) اجمع (الاسم والجملة والنتيجة قبل وبعد) في قائمة أثناء العمل، فأنت تحتاج الأربعة
#    لجدول الأثر، وإعادة تجميعها لاحقًا هي كيف تصير خاطئة.
# ابحث عن: "pandas groupby transform mean"
# ────────────────────────────────────────────────────────────────────

engineered = cleaned.copy()
def record(name, sentence):
    """Add `name` to the model, refit, and store the before/after pair."""
    # TODO: the new score so the next feature is measured against it.
    # مهمة: ليُقاس عليها ما بعدها.
# TODO: Feature 1 — the order's value before any discount.
# مهمة: الخاصية الأولى — قيمة الطلب قبل أي خصم.
# TODO: Feature 2 — the discount in money rather than as a percentage.
# مهمة: الخاصية الثانية — الخصم بالمال لا بالنسبة المئوية.
# TODO: Feature 3 — this order's unit price relative to its own city's average.
# مهمة: الخاصية الثالثة — سعر وحدة هذا الطلب بالنسبة إلى متوسط مدينته.
# TODO: Feature 4 — how many days passed between the order and the shipment.
# مهمة: الخاصية الرابعة — كم يومًا مضى بين الطلب والشحن.
# TODO: Feature 5 — yours. Say the sentence in the markdown cell below before you look.
# مهمة: الخاصية الخامسة — من ابتكارك. قل جملتها في خلية الشرح أدناه قبل أن تنظر.
lift = pd.DataFrame(lift_rows)
print(lift[["feature", "before", "after", "delta"]].to_string(
    index=False, float_format=lambda v: f"{v:+.6f}"))
print(f"\nbaseline {baseline_r2:.4f}  ->  final {lift['after'].iloc[-1]:.4f}")

**Your feature 5, and its sentence:** _(what did you build, what did you believe, and did the
measurement agree?)_

<div dir="rtl" align="right">

**خاصيتك الخامسة وجملتها:** _(ماذا بنيت، وما اعتقدته، وهل وافقك القياس؟)_

</div>

Read that table again, because it is the most useful thing in the lab.

**One feature was worth +0.19.** `gross_order_value` — `unit_price × quantity` — took the model from
0.79 to 0.98. And it is not a clever feature. It is the multiplication a linear model **cannot do for
itself**: given `unit_price` and `quantity` as separate columns, a linear model can only add weighted
versions of them, and revenue is a product. Handing it the product is the entire lift.

**The second was worth +0.008.** `discount_amount` is `gross × discount_pct`, another product the
model could not form on its own.

**The other three were worth nothing** — deltas in the fifth and sixth decimal, one of them negative.
They were not stupid ideas. A price relative to its city, a shipping delay, a bulk-order flag: all
four sentences are plausible, and a reasonable analyst would try all of them. The data says they
carry nothing once you already know the order's gross value and its discount.

That is what feature engineering actually looks like. One or two ideas do the work, several do
nothing, and **you cannot tell which is which by reasoning about them.** The only way to find out is
one at a time, with a number after each.

And now the connection back to yesterday. Dropping `commission_paid` cost you 0.21. Feature 1 just
recovered 0.19 of it — from `unit_price` and `quantity`, two columns that exist the moment the order
is placed. The commission was the same information arriving from the future. This is the honest route
to it, and it is worth noticing that the honest route got you almost all the way.

<div dir="rtl" align="right">

اقرأ ذلك الجدول مرة أخرى، فهو أنفع ما في المعمل.

**خاصية واحدة ساوت +٠٫١٩.** فالعمود `gross_order_value`، أي `unit_price × quantity`، نقل النموذج من
٠٫٧٩ إلى ٠٫٩٨. وهو ليس خاصية ذكية، بل هو الضرب الذي **لا يستطيع النموذج الخطّي أن يفعله بنفسه**: فإذا
أُعطي سعر الوحدة والكمية عمودين منفصلين لم يستطع إلا جمع نسختين مرجّحتين منهما، والإيراد حاصل ضرب.
فإعطاؤه الحاصل هو الزيادة كلها.

**والثانية ساوت +٠٫٠٠٨.** فالعمود `discount_amount` هو `gross × discount_pct`، وهو حاصل ضرب آخر لم
يستطع النموذج تشكيله بنفسه.

**والثلاث الأخرى لم تساوِ شيئًا** — فروق في المنزلة الخامسة والسادسة، وإحداها سالبة. ولم تكن أفكارًا
سخيفة: فالسعر بالنسبة إلى مدينته، وتأخّر الشحن، ومؤشّر الطلب الكبير — الجمل الثلاث معقولة، وأي محلّل
معقول سيجرّبها. لكن البيانات تقول إنها لا تحمل شيئًا بعد أن تعرف قيمة الطلب الإجمالية وخصمه.

وهذا هو شكل هندسة الخصائص فعلًا: فكرة أو فكرتان تقومان بالعمل، وعدّة أفكار لا تفعل شيئًا،
و**لا تستطيع التمييز بينها بالتفكير فيها.** والسبيل الوحيد للمعرفة هو واحدة واحدة، ورقم بعد كل واحدة.

والآن الصلة بالأمس. كلّفك حذف `commission_paid` مقدار ٠٫٢١، وقد استعادت الخاصية الأولى ٠٫١٩ منه — من
`unit_price` و`quantity`، وهما عمودان موجودان لحظة تقديم الطلب. فالعمولة كانت المعلومة نفسها قادمةً
من المستقبل، وهذا هو الطريق الصادق إليها، ويستحق أن تلاحظ أن الطريق الصادق أوصلك إلى معظمها.

</div>

### Task 2.3 — write the lift table

`lift_table.md`: one row per feature, with the hypothesis, the score before, the score after, and the
delta. Five rows.

**Keep the negative and the zero deltas.** This is the instruction most likely to be quietly ignored,
so here is the reason it matters: a table containing only the features that worked tells a reader you
had two good ideas. A table containing all five tells them you had five ideas and *measured* them,
which is a much stronger claim about your work and the only one that is true.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — اكتب جدول الأثر

الملف `lift_table.md`: صف لكل خاصية، مع الفرضية والنتيجة قبل والنتيجة بعد والفرق. خمسة صفوف.

**واحتفظ بالفروق السالبة والصفرية.** وهذه أكثر التعليمات عرضةً للتجاهل بهدوء، فهذا سبب أهميّتها:
الجدول الذي يحتوي الخصائص الناجحة فقط يقول لقارئه إنك أتيت بفكرتين جيّدتين، والجدول الذي يحتوي الخمس
كلها يقول له إنك أتيت بخمس أفكار و**قِستها**، وهذا ادعاء أقوى بكثير عن عملك وهو الوحيد الصحيح.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Turn `lift` into a markdown table. Five rows, and a header row.
# 2) Six decimal places, not two — three of your deltas live in the fifth decimal
#    and rounding them to 0.00 destroys the finding.
# 3) Add a line saying which feature carried the lift and which carried nothing.
# 4) Save it to ARTEFACT_DIR and print it back so you can see what you wrote.
# Search: "pandas to_markdown"
#
# ١) حوّل `lift` إلى جدول ماركداون: خمسة صفوف وصف رأس.
# ٢) بستّ منازل عشرية لا اثنتين، فثلاثة من فروقك تقع في المنزلة الخامسة
#    وتقريبها إلى ٠٫٠٠ يُلغي النتيجة.
# ٣) أضف سطرًا يقول أي خاصية حملت الزيادة وأيّها لم تحمل شيئًا.
# ٤) احفظه في ARTEFACT_DIR واطبعه بعده لترى ما كتبته.
# ابحث عن: "pandas to_markdown"
# ────────────────────────────────────────────────────────────────────

# TODO: Write lift_table.md — five feature rows, six decimals, negatives kept.
# مهمة: اكتب ملف lift_table.md بخمسة صفوف وستّ منازل عشرية، مع الاحتفاظ بالسالب.

### Task 2.4 — three scalers, one linear model

Scale the whole feature matrix three ways — Min-Max, Standard, Robust — and refit the linear model on
each.

**Fit the scaler on the training rows only, and transform both halves with it.** The scaler learns
statistics — a min, a max, a mean, a median — and those statistics are information about the data. If
it learns them from the test rows too, the test set has helped build the transformation and stopped
being held out. This is the single most important mechanical rule of the session and W2D5 spends its
whole slot on why.

Predict what happens to the scores before you run it.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — ثلاثة مُقيِّسات ونموذج خطّي واحد

قيّس مصفوفة الخصائص كلها بثلاث طرائق — Min-Max وStandard وRobust — وأعد تدريب النموذج الخطّي على كل
واحدة.

**ودرّب المُقيِّس على صفوف التدريب وحدها، وحوّل النصفين به.** فالمُقيِّس يتعلّم مقاييس إحصائية — أصغر
قيمة وأكبر قيمة ومتوسطًا ووسيطًا — وهذه المقاييس معلومات عن البيانات. فإذا تعلّمها من صفوف الاختبار
أيضًا فقد شاركت مجموعة الاختبار في بناء التحويل وتوقّفت عن كونها محجوزة. وهذه أهمّ قاعدة عمليّة في
الجلسة، ويُخصّص اليوم الخامس وقته كله لبيان السبب.

توقّع ما سيحدث للنتائج قبل التشغيل.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Split once, then for each scaler: fit on X_train, transform X_train AND X_test
#    with that fitted scaler. Never fit on the test rows.
# 2) Fit the same LinearRegression on each scaled training set and score on the
#    matching scaled test set.
# 3) Print all four numbers — unscaled and the three scalers — to six decimals.
# 4) Then keep the fitted Standard scaler and its statistics; task 2.5 and the
#    sanity check both need it.
# Search: "sklearn scaler fit_transform train transform test"
# https://scikit-learn.org/stable/modules/preprocessing.html
#
# ١) قسّم مرة واحدة، ثم لكل مُقيِّس: درّبه على X_train وحوّل به X_train **و**X_test.
#    ولا تدرّبه على صفوف الاختبار أبدًا.
# ٢) درّب الانحدار الخطّي نفسه على كل مجموعة تدريب مُقيَّسة وقيّم على مجموعة
#    الاختبار المُقيَّسة المقابلة.
# ٣) اطبع الأرقام الأربعة — غير المُقيَّس والمُقيِّسات الثلاثة — بستّ منازل عشرية.
# ٤) ثم احتفظ بمُقيِّس Standard المُدرَّب وبمقاييسه، فالمهمة الخامسة وفحص النتائج
#    يحتاجانه.
# ابحث عن: "sklearn scaler fit_transform train transform test"
# ────────────────────────────────────────────────────────────────────

X_train, X_test, y_train, y_test = split(engineered, numeric)
print(f"unscaled            R2 = {LinearRegression().fit(X_train, y_train).score(X_test, y_test):.6f}")
scalers = {"Min-Max": MinMaxScaler(), "Standard": StandardScaler(), "Robust": RobustScaler()}
scaled_scores = {}
# TODO: For each scaler: fit on the TRAINING rows only, transform both halves, refit, score.
# مهمة: لكل مُقيِّس: درّبه على صفوف **التدريب** وحدها، وحوّل النصفين، وأعد التدريب، وقيّم.
standard = StandardScaler().fit(X_train)

All four numbers are the same to six decimal places.

That is the correct answer, and it is not a disappointment. `LinearRegression` here is solved by
ordinary least squares, and an affine rescaling of a column is absorbed exactly into its coefficient:
halve the column, double the weight, identical predictions. So for **this** model, scaling changes
nothing about the fit.

Which raises the obvious question: then why is half the session about scalers? Because the models
where it matters are the ones you meet next:

- **Distance-based** — K-Means, K-Nearest-Neighbours — measure with Euclidean distance, and you saw
  in Section 1 what unscaled columns do to a distance. W1D5's clustering lab was damaged by exactly
  this.
- **Gradient-based** — anything trained by gradient descent, which from week 3 onwards is everything
  — takes one shared learning rate across wildly different column scales, and either crawls or
  diverges. The whole of week 3 depends on this cell being understood.
- **Regularised** — Ridge and Lasso, tomorrow — penalise coefficients by size, so a column measured
  in thousands gets a small coefficient and is penalised less than a column measured in units. The
  penalty stops meaning what you think it means.

The task here was not to find a winner. It was to notice that there is nothing to win, and to be able
to say why.

<div dir="rtl" align="right">

الأرقام الأربعة متساوية إلى ستّ منازل عشرية.

وهذا هو الجواب الصحيح وليس خيبة أمل. فالانحدار الخطّي هنا يُحلّ بأصغر المربّعات، والتقييس الخطّي للعمود
يُمتصّ بالكامل في مُعامله: فإذا نصّفت العمود وضاعفت الوزن كانت التنبّؤات متطابقة. فبالنسبة إلى **هذا**
النموذج لا يغيّر التقييس شيئًا في المطابقة.

ويطرح ذلك سؤالًا بديهيًا: فلماذا نصف الجلسة عن المُقيِّسات؟ لأن النماذج التي يهمّها الأمر هي التي
ستقابلها بعد قليل:

- **القائمة على المسافة** — مثل K-Means والجيران الأقرب — تقيس بالمسافة الإقليدية، وقد رأيت في القسم
  الأول ما تفعله الأعمدة غير المُقيَّسة بالمسافة. ومعمل التجميع في الأسبوع الأول اليوم الخامس تضرّر
  بهذا بالضبط.
- **القائمة على التدرّج** — أي كل ما يُدرَّب بالنزول التدريجي، وهو من الأسبوع الثالث فصاعدًا كل شيء —
  تأخذ معدّل تعلّم واحدًا مشتركًا على أعمدة متباعدة المقاييس، فإمّا تزحف وإمّا تتباعد. والأسبوع الثالث
  كله يتوقّف على فهم هذه الخليّة.
- **المُنظَّمة** — مثل Ridge وLasso غدًا — تعاقب المعاملات بحجمها، فالعمود المقيس بالآلاف يأخذ معاملًا
  صغيرًا فيُعاقَب أقل من العمود المقيس بالوحدات. فتتوقّف العقوبة عن أن تعني ما تظنّه.

والمهمة هنا لم تكن إيجاد فائز، بل ملاحظة أن لا شيء يُفاز به، والقدرة على قول السبب.

</div>

### Task 2.5 — the same question, asked of a tree

Now fit a `RandomForestRegressor` twice: once on the unscaled features, once on the standard-scaled
ones. Same seed, same split, same everything else.

The scores will be the same. Before you run it, write down why — and "because trees are robust" is
not the answer. The answer is one sentence about what a tree actually *does* with a column.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — السؤال نفسه مطروحًا على شجرة

درّب الآن `RandomForestRegressor` مرتين: مرة على الخصائص غير المُقيَّسة ومرة على المُقيَّسة معياريًا.
بالبذرة نفسها والتقسيم نفسه وكل شيء آخر نفسه.

وستتساوى النتيجتان. واكتب قبل التشغيل لماذا — وقولك «لأن الشجرة قويّة» ليس جوابًا. والجواب جملة واحدة
عمّا **تفعله** الشجرة بالعمود فعلًا.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Fit a RandomForestRegressor with n_estimators=N_TREES and random_state=42 on
#    the unscaled training data, and score it.
# 2) Do the same on the standard-scaled training data, using the scaler you already
#    fitted on train in task 2.4.
# 3) Print both to SIX decimals and their difference. Six, not four — the story is
#    in whether the last two digits move.
# Search: "sklearn RandomForestRegressor feature scaling"
# https://scikit-learn.org/stable/modules/tree.html
#
# ١) درّب RandomForestRegressor بعدد أشجار N_TREES وبذرة ٤٢ على بيانات التدريب غير
#    المُقيَّسة، وقيّمه.
# ٢) وافعل ذلك على بيانات التدريب المُقيَّسة معياريًا، بالمُقيِّس الذي دربته على
#    التدريب في المهمة الرابعة.
# ٣) اطبع الرقمين بستّ منازل عشرية والفرق بينهما. ستّ لا أربع، فالقصة في هل
#    تتحرّك المنزلتان الأخيرتان.
# ابحث عن: "sklearn RandomForestRegressor feature scaling"
# ────────────────────────────────────────────────────────────────────

# TODO: Fit the forest on the unscaled training data and score it.
# مهمة: درّب الغابة على بيانات التدريب غير المُقيَّسة وقيّمها.
# TODO: Fit an identical forest on the standard-scaled data and score it.
# مهمة: درّب غابة مطابقة على البيانات المُقيَّسة معياريًا وقيّمها.
print(f"forest, unscaled  R2 = {tree_unscaled:.6f}")
print(f"forest, scaled    R2 = {tree_scaled:.6f}")
print(f"difference             {abs(tree_unscaled - tree_scaled):.2e}")

The two scores agree to four decimal places, and they differ in the fifth.

**Why they agree:** a tree does not measure distances or add weighted columns. It asks *"is this
column above some threshold?"* and splits the rows into two groups. Standard-scaling is monotonic —
it never changes which rows are above which other rows — so every threshold the tree could have
chosen still exists, just relabelled. Same splits, same groups, same predictions. Scaling a tree's
inputs is work that produces nothing.

**Why they differ at all:** floating-point arithmetic. The scaled column has different rounding
behaviour, so a handful of candidate thresholds land on the other side of a tie, and a few of the 60
trees make a marginally different split. The difference is 1e-5, which is noise. If you had expected
*bit-identical* results, that expectation was the thing that was wrong — not the tree.

Two consequences worth carrying:

1. Do not scale for a tree, a random forest, or a gradient-boosted model. It cannot help, and it
   costs you a fitted object to keep track of and get wrong at prediction time.
2. If you ever scale a tree's input and the score changes **meaningfully**, you have a bug — most
   likely a scaler fitted on the wrong rows.

<div dir="rtl" align="right">

تتفق النتيجتان إلى أربع منازل عشرية وتختلفان في الخامسة.

**سبب الاتفاق:** أن الشجرة لا تقيس مسافات ولا تجمع أعمدة مرجّحة، بل تسأل *«هل هذا العمود أعلى من عتبة
ما؟»* وتقسم الصفوف مجموعتين. والتقييس المعياري رتيب لا يغيّر أبدًا أي الصفوف أعلى من غيرها، فكل عتبة
كانت الشجرة تستطيع اختيارها لا تزال موجودة وإنما بتسمية أخرى. فالتقسيمات نفسها والمجموعات نفسها
والتنبّؤات نفسها. وتقييس مُدخلات الشجرة عمل لا يُنتج شيئًا.

**وسبب اختلافهما أصلًا:** حساب الفاصلة العائمة. فللعمود المُقيَّس سلوك تقريب مختلف، فتقع حفنة من
العتبات المرشّحة في الجهة الأخرى من التعادل، وتتّخذ بضع أشجار من الستّين تقسيمًا مختلفًا اختلافًا
هامشيًا. والفرق من رتبة ١٠ أُس -٥، وهذا ضوضاء. وإن كنت توقّعت نتائج متطابقة **بتًّا ببت** فذلك التوقّع
هو الخطأ، لا الشجرة.

ونتيجتان تستحقّان الحمل:

١. لا تُقيّس لشجرة ولا لغابة عشوائية ولا لنموذج تعزيز تدريجي. فلا يمكن أن يساعد، ويكلّفك كائنًا
   مُدرَّبًا تتابعه وتخطئ فيه وقت التنبّؤ.
٢. وإن قيّست مُدخل شجرة يومًا فتغيّرت النتيجة تغيّرًا **ذا معنى**، فعندك عيب — وأرجحه مُقيِّس مُدرَّب
   على الصفوف الخاطئة.

</div>

## Section 3 — Stretch: try to break your best feature  (≈30 min)

Open-ended. Lower expectation of completeness — get through the first part, and treat the second as
homework if you run out of time.

`gross_order_value` was worth +0.19. Now attack it, with the same question that condemned
`commission_paid` yesterday:

> **For a brand-new order that has just been placed, can you fill this column in?**

Work through it properly rather than answering from memory:

1. Name the columns it is built from. For each one, say **when** it becomes known — at order time, at
   shipping time, at settlement, or later.
2. If every input is known at order time, the feature is admissible. Write that down with the
   reasoning, not just the verdict.
3. Then do the same audit for the other four. `unit_price_vs_city` is the interesting one: it uses
   the **mean unit price of the city**, computed across the whole dataset. Is that mean available for
   a new order? What exactly would you have to do differently to compute it honestly, and does the
   answer change if a new city appears? (This is the same shape as yesterday's target encoding, and
   tomorrow's `Pipeline` is the mechanism that fixes it.)

Then the harder question, which has no clean answer:

> Feature 1 recovered 0.19 of the 0.21 that dropping the commission cost. **Is the model now as good
> as the leaking one was?**

It scores 0.98 against 1.00, so numerically almost. Argue the other side: what is different about
those two 0.98-ish numbers, and which one would you defend in a meeting? Write the paragraph.

**Link to your capstone:** every engineered feature in your capstone needs this audit, and the report
rubric asks for it explicitly. A feature that quietly depends on something known only after the
prediction is the most common way a capstone's headline number turns out to be fiction.

<div dir="rtl" align="right">

## القسم الثالث — الإضافي: حاول أن تُكسِر أفضل خصائصك (نحو ٣٠ دقيقة)

قسم مفتوح، ولا يُتوقّع إكماله بالكامل — أنجز الجزء الأول واعتبر الثاني واجبًا منزليًا إن ضاق الوقت.

ساوى `gross_order_value` مقدار +٠٫١٩. فهاجمه الآن بالسؤال نفسه الذي أدان `commission_paid` بالأمس:

> **لطلب جديد قُدِّم الآن، هل تستطيع ملء هذا العمود؟**

واعمل على السؤال عملًا صحيحًا لا إجابةً من الذاكرة:

١. سمِّ الأعمدة التي بُني منها، وقل لكل واحد **متى** يُعرف: وقت الطلب أم وقت الشحن أم عند التسوية أم بعد ذلك.
٢. فإن كان كل مُدخل معروفًا وقت الطلب فالخاصية مقبولة. واكتب ذلك مع التعليل لا الحكم وحده.
٣. ثم افحص الأربع الأخرى الفحص نفسه. والمثيرة هي `unit_price_vs_city` فهي تستخدم **متوسط سعر الوحدة
   في المدينة** محسوبًا على البيانات كلها. فهل هذا المتوسط متاح لطلب جديد؟ وما الذي كان يجب أن تفعله
   بدقّة لحسابه حسابًا صادقًا، وهل يتغيّر الجواب إذا ظهرت مدينة جديدة؟ (وهذا هو شكل الترميز بالهدف
   بالأمس نفسه، وخطّ المعالجة `Pipeline` غدًا هو الآليّة التي تُصلحه.)

ثم السؤال الأصعب الذي لا إجابة نظيفة له:

> استعادت الخاصية الأولى ٠٫١٩ من الـ ٠٫٢١ التي كلّفها حذف العمولة. **فهل صار النموذج بجودة النموذج
> المُسرِّب؟**

نتيجته ٠٫٩٨ مقابل ١٫٠٠، فهو رقميًا يكاد. فحاجج الجهة الأخرى: ما الفرق بين هذين الرقمين المتقاربين،
وأيّهما ستدافع عنه في اجتماع؟ اكتب الفقرة.

**الصلة بمشروعك:** كل خاصية مُهندَسة في مشروعك تحتاج هذا الفحص، وكرّاسة تقييم التقرير تطلبه صراحةً.
فالخاصية التي تعتمد بهدوء على شيء لا يُعرف إلا بعد التنبّؤ هي أشيع طريقة يتحوّل بها الرقم الرئيسي في
مشروع تخرّج إلى خيال.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Build a small table: feature, the columns it depends on, and when each of those
#    is known. Four values will do: order time, shipping, settlement, never.
# 2) Mark each feature admissible or not, and say why in one phrase.
# 3) For unit_price_vs_city, actually check the damage: recompute the city means on
#    the training rows only, re-score, and see whether the number moved.
# 4) Then write your paragraph in the markdown cell below.
# Search: "data leakage feature availability at prediction time"
#
# ١) ابنِ جدولًا صغيرًا: الخاصية، والأعمدة التي تعتمد عليها، ومتى يُعرف كل منها.
#    وتكفي أربع قيم: وقت الطلب، والشحن، والتسوية، وأبدًا.
# ٢) علّم كل خاصية بمقبولة أو غير مقبولة، وقل السبب في عبارة واحدة.
# ٣) وللخاصية unit_price_vs_city افحص الضرر فعلًا: أعد حساب متوسطات المدن على صفوف
#    التدريب وحدها، وأعد التقييم، وانظر هل تحرّك الرقم.
# ٤) ثم اكتب فقرتك في خلية الشرح أدناه.
# ابحث عن: "data leakage feature availability at prediction time"
# ────────────────────────────────────────────────────────────────────

# TODO: Audit all five features for availability at prediction time.
# مهمة: افحص الخصائص الخمس كلها من جهة توفّرها وقت التنبّؤ.
# TODO: Recompute unit_price_vs_city using training rows only, and see if the score moved.
# مهمة: أعد حساب unit_price_vs_city بصفوف التدريب وحدها وانظر هل تحرّك الرقم.

**Your audit and your paragraph:** _(which features are admissible and why; then — is a legitimate
0.98 the same as a leaking 1.00, and which would you defend?)_

<div dir="rtl" align="right">

**فحصك وفقرتك:** _(أي الخصائص مقبولة ولماذا؟ ثم: هل ٠٫٩٨ المشروعة كـ ١٫٠٠ المُسرِّبة، وأيّهما ستدافع
عنه؟)_

</div>

## Save your artefact

`engineered.parquet` — the cleaned frame plus your five features, exactly five columns wider than
what you loaded.

Note what is **not** saved: the scalers. A fitted scaler is not data, it is part of the model, and
saving it beside the table is how it drifts out of sync with the model that needs it. Tomorrow you
put the scaler *inside* a `Pipeline`, where it belongs and where it cannot be forgotten.

One column in the file is there for you to argue with. `shipping_days` failed the availability audit
in Section 3 — a shipping date does not exist when the order is placed — and it is saved anyway,
because deciding what to do about it is your stretch task and not something this notebook should
settle for you. It is worth nothing to the model (+0.000002), so dropping it costs you nothing; what
it costs to *keep* is a model that cannot be served. Nothing downstream in this week loads this
file, so the decision is yours to make and to write down.

<div dir="rtl" align="right">

## احفظ مخرجاتك

الملف `engineered.parquet` هو الجدول المُنظَّف مع خصائصك الخمس، أوسع بخمسة أعمدة بالضبط من الذي حمّلته.

ولاحظ ما **لا** يُحفَظ: المُقيِّسات. فالمُقيِّس المُدرَّب ليس بيانات بل جزء من النموذج، وحفظه بجانب
الجدول هو كيف يفترق عن النموذج الذي يحتاجه. وغدًا تضع المُقيِّس **داخل** خطّ معالجة `Pipeline`، وهو
موضعه، وهناك لا يمكن نسيانه.

وفي الملف عمود واحد موجود لتُحاججه. فالعمود `shipping_days` لم ينجح في فحص التوفّر في القسم الثالث —
إذ لا يوجد تاريخ شحن لحظة تقديم الطلب — وهو محفوظ مع ذلك، لأن القرار فيه مهمّتك الإضافية لا شيء
ينبغي لهذا الدفتر أن يحسمه عنك. وهو لا يساوي شيئًا للنموذج (+٠٫٠٠٠٠٠٢) فحذفه لا يكلّفك شيئًا، أما
**إبقاؤه** فيكلّفك نموذجًا لا يمكن تشغيله. ولا يُحمّل هذا الملف شيء لاحق في هذا الأسبوع، فالقرار قرارك
وعليك كتابته.

</div>

In [ ]:
out = ARTEFACT_DIR / "engineered.parquet"
engineered.to_parquet(out, index=False)

reloaded = pd.read_parquet(out)
print(f"Saved {out}")
print(f"{reloaded.shape[0]:,} rows x {reloaded.shape[1]} columns "
      f"({reloaded.shape[1] - cleaned.shape[1]} more than cleaned.parquet)")
print(f"new columns: {[c for c in reloaded.columns if c not in cleaned.columns]}")
print(f"round-trips to an equal DataFrame: {reloaded.equals(engineered)}")

## Sanity check

Run this last. Every check that fails tells you what to fix and why.

The fourth one is the lesson of Section 2.5 written as an assertion: if the forest's scaled and
unscaled scores ever stop matching, something is wrong with your scaler and not with the tree.

<div dir="rtl" align="right">

## فحص النتائج

شغّل هذه الخلية أخيرًا. كل فحص يفشل يخبرك بما يجب إصلاحه ولماذا.

والفحص الرابع هو درس القسم ٢٫٥ مكتوبًا تحقّقًا: فإن توقّفت نتيجتا الغابة المُقيَّسة وغير المُقيَّسة
عن التطابق يومًا فالخلل في مُقيِّسك لا في الشجرة.

</div>

In [ ]:
# --- Sanity checks ----------------------------------------------------------------

new_columns = [c for c in engineered.columns if c not in cleaned.columns]

check(len(new_columns) == 5,
      f"the engineered frame should have exactly 5 more columns than cleaned.parquet, "
      f"got {len(new_columns)}: {new_columns}",
      f"يجب أن يزيد الجدول المُهندَس بخمسة أعمدة بالضبط عن cleaned.parquet، "
      f"والزيادة {len(new_columns)}: {new_columns}")

degenerate = [c for c in new_columns
              if engineered[c].isna().all() or engineered[c].nunique() <= 1]
check(not degenerate,
      f"no engineered column may be entirely NaN or constant — these are: {degenerate}",
      f"لا يجوز أن يكون أي عمود مُهندَس كله NaN أو ثابتًا — وهذه هي: {degenerate}")

check(len(lift) >= 5 and lift["delta"].notna().all(),
      f"the lift table needs 5 feature rows each with a measured delta, got {len(lift)}",
      f"يحتاج جدول الأثر خمسة صفوف لكل واحد فرق مقيس، والموجود {len(lift)}")

check(np.isclose(tree_unscaled, tree_scaled, atol=1e-4),
      f"the forest's scaled and unscaled R2 must match to 4 decimals — a tree splits on "
      f"thresholds, and scaling is monotonic, so nothing should change. Got "
      f"{tree_unscaled:.6f} vs {tree_scaled:.6f}",
      f"يجب أن تتطابق نتيجتا الغابة المُقيَّسة وغير المُقيَّسة إلى أربع منازل — فالشجرة تقسم "
      f"على عتبات والتقييس رتيب، فلا ينبغي أن يتغيّر شيء. والناتج "
      f"{tree_unscaled:.6f} مقابل {tree_scaled:.6f}")

# The scaler was fitted on train only, so the TEST rows must not come out with mean 0 and
# std 1 — only the training rows do. If the test statistics are also perfect, the scaler
# saw the test set, and this is the check that catches it.
test_scaled_mean = float(np.abs(standard.transform(X_test).mean(axis=0)).max())
train_scaled_mean = float(np.abs(standard.transform(X_train).mean(axis=0)).max())

check(train_scaled_mean < 1e-9 < test_scaled_mean,
      f"the scaler must be fitted on the training rows only: the train columns should come "
      f"out centred (max |mean| {train_scaled_mean:.2e}) and the test columns should NOT "
      f"(max |mean| {test_scaled_mean:.2e}). If both are ~0 you fitted on everything.",
      f"يجب تدريب المُقيِّس على صفوف التدريب وحدها: فأعمدة التدريب تخرج مُركَّزة "
      f"(أقصى قيمة مطلقة للمتوسط {train_scaled_mean:.2e}) وأعمدة الاختبار **لا** "
      f"(أقصاها {test_scaled_mean:.2e}). فإن كانا قريبين من الصفر فقد دربته على الكل.")

lift_sections = (ARTEFACT_DIR / "lift_table.md").read_text(encoding="utf-8")
check(lift_sections.count("|") >= 5 * 6,
      f"lift_table.md should contain a 5-row markdown table, found "
      f"{lift_sections.count('|')} pipe characters",
      f"يجب أن يحتوي ملف lift_table.md جدولًا من خمسة صفوف، والموجود "
      f"{lift_sections.count('|')} فاصلًا رأسيًا")

report()

## What's next

Tomorrow (**W2D5**) is the most consequential lab of the week, and it starts by taking away the thing
every score you have read so far depended on: **the single train/test split.**

You have compared 0.7915 against 0.7885 and drawn a conclusion. Tomorrow you will fit the same model
on five different splits and see how far apart those five numbers are — and then you will know
whether a gap of 0.003 was ever a result at all.

Then you build a `Pipeline`, which is where the scaler you fitted by hand today belongs, and then you
break it on purpose to see what leakage looks like when it is a single line in the wrong order.

<div dir="rtl" align="right">

## ماذا بعد

غدًا (**الأسبوع ٢ اليوم ٥**) هو أهمّ معامل الأسبوع، ويبدأ بأن يسحب منك ما اعتمدت عليه كل نتيجة قرأتها
حتى الآن: **التقسيم الواحد إلى تدريب واختبار.**

فقد قارنت ٠٫٧٩١٥ بـ ٠٫٧٨٨٥ وخرجت باستنتاج. وغدًا تُدرّب النموذج نفسه على خمسة تقسيمات مختلفة وترى كم
تتباعد تلك الأرقام الخمسة — ثم تعرف هل كان فرق ٠٫٠٠٣ نتيجةً أصلًا.

ثم تبني خطّ معالجة `Pipeline` وهو موضع المُقيِّس الذي دربته يدويًا اليوم، ثم تُكسِره عن قصد لترى كيف
يبدو التسريب حين يكون سطرًا واحدًا في الترتيب الخاطئ.

</div>